In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("lab2-etl")
    .getOrCreate()
)

jdbc_url = "jdbc:postgresql://postgres_db:5432/bigdata_warehouse"
connection_props = {
    "user": "warehouse_user",
    "password": "secure_pass_2026",
    "driver": "org.postgresql.Driver"
}

In [ ]:
df = spark.read.jdbc(
    url=jdbc_url,
    table="mock_data",
    properties=connection_props
)

df.printSchema()

Читаем независимые измерения и записываем в таблицы

In [ ]:
dim_customer = (
    df.select(
        col("customer_first_name").alias("first_name"),
        col("customer_last_name").alias("last_name"),
        col("customer_age").alias("age"),
        col("customer_email").alias("email"),
        col("customer_country").alias("country"),
        col("customer_postal_code").alias("postal_code"),
        col("customer_pet_type").alias("pet_type"),
        col("customer_pet_name").alias("pet_name"),
        col("customer_pet_breed").alias("pet_breed")
    )
    .distinct()
)

In [ ]:
dim_seller = (
    df.select(
        col("seller_first_name").alias("first_name"),
        col("seller_last_name").alias("last_name"),
        col("seller_email").alias("email"),
        col("seller_country").alias("country"),
        col("seller_postal_code").alias("postal_code")
    )
    .distinct()
)

In [ ]:
dim_store = (
    df.select(
        col("store_name").alias("name"),
        col("store_location").alias("location"),
        col("store_city").alias("city"),
        col("store_state").alias("state"),
        col("store_country").alias("country"),
        col("store_phone").alias("phone"),
        col("store_email").alias("email")
    )
    .distinct()
)

In [ ]:
dim_supplier = (
    df.select(
        col("supplier_name").alias("name"),
        col("supplier_contact").alias("contact"),
        col("supplier_email").alias("email"),
        col("supplier_phone").alias("phone"),
        col("supplier_address").alias("address"),
        col("supplier_city").alias("city"),
        col("supplier_country").alias("country")
    )
    .distinct()
)

In [ ]:
dim_customer.write.jdbc(
    url=jdbc_url,
    table="dim_customer",
    mode="append",
    properties=connection_props
)

dim_seller.write.jdbc(
    url=jdbc_url,
    table="dim_seller",
    mode="append",
    properties=connection_props
)

dim_store.write.jdbc(
    url=jdbc_url,
    table="dim_store",
    mode="append",
    properties=connection_props
)

dim_supplier.write.jdbc(
    url=jdbc_url,
    table="dim_supplier",
    mode="append",
    properties=connection_props
)

Читаем заполненные таблицы и записываем product

In [ ]:
store_df = spark.read.jdbc(
    url=jdbc_url,
    table="dim_store",
    properties=connection_props
)

supplier_df = spark.read.jdbc(
    url=jdbc_url,
    table="dim_supplier",
    properties=connection_props
)

In [ ]:
dim_product = (
    df.alias("m")
    .join(
        store_df.alias("st"),
        (col("m.store_name") == col("st.name")) &
        (col("m.store_location") == col("st.location")) &
        (col("m.store_city") == col("st.city")) &
        col("m.store_state").eqNullSafe(col("st.state"))  &
        (col("m.store_country") == col("st.country")) &
        (col("m.store_phone") == col("st.phone")) &
        (col("m.store_email") == col("st.email")),
        "left"
    )
    .join(
        supplier_df.alias("sp"),
        (col("m.supplier_name") == col("sp.name")) &
        (col("m.supplier_contact") == col("sp.contact")) &
        (col("m.supplier_email") == col("sp.email")) &
        (col("m.supplier_phone") == col("sp.phone")) &
        (col("m.supplier_address") == col("sp.address")) &
        (col("m.supplier_city") == col("sp.city")) &
        (col("m.supplier_country") == col("sp.country")),
        "left"
    )
    .select(
        col("m.product_name").alias("product_name"),
        col("m.product_category").alias("product_category"),
        col("m.product_price").alias("product_price"),
        col("m.product_quantity").alias("product_quantity"),
        col("m.pet_category").alias("pet_category"),
        col("m.product_weight").alias("product_weight"),
        col("m.product_color").alias("product_color"),
        col("m.product_size").alias("product_size"),
        col("m.product_brand").alias("product_brand"),
        col("m.product_material").alias("product_material"),
        col("m.product_description").alias("product_description"),
        col("m.product_rating").alias("product_rating"),
        col("m.product_reviews").alias("product_reviews"),
        col("m.product_release_date").alias("product_release_date"),
        col("m.product_expiry_date").alias("product_expiry_date"),
        col("st.id").alias("store_id"),
        col("sp.id").alias("supplier_id")
    )
    .distinct()
)

In [ ]:
dim_product.write.jdbc(
    url=jdbc_url,
    table="dim_product",
    mode="append",
    properties=connection_props
)

Читаем таблицы и пишем product

In [ ]:
customer_df = spark.read.jdbc(
    url=jdbc_url,
    table="dim_customer",
    properties=connection_props
)

seller_df = spark.read.jdbc(
    url=jdbc_url,
    table="dim_seller",
    properties=connection_props
)

product_df = spark.read.jdbc(
    url=jdbc_url,
    table="dim_product",
    properties=connection_props
)

In [ ]:
fact_sales = (
    df.alias("m")
    .join(
        customer_df.alias("c"),
        (col("m.customer_first_name") == col("c.first_name")) &
        (col("m.customer_last_name") == col("c.last_name")) &
        (col("m.customer_age") == col("c.age")) &
        (col("m.customer_email") == col("c.email")) &
        (col("m.customer_country") == col("c.country")) &
        (col("m.customer_postal_code").eqNullSafe(col("c.postal_code"))) &
        (col("m.customer_pet_type") == col("c.pet_type")) &
        (col("m.customer_pet_name") == col("c.pet_name")) &
        (col("m.customer_pet_breed") == col("c.pet_breed")),
        "left"
    )
    .join(
        seller_df.alias("s"),
        (col("m.seller_first_name") == col("s.first_name")) &
        (col("m.seller_last_name") == col("s.last_name")) &
        (col("m.seller_email") == col("s.email")) &
        (col("m.seller_country") == col("s.country")) &
        (col("m.seller_postal_code").eqNullSafe(col("s.postal_code"))),
        "left"
    )
    .join(
        product_df.alias("p"),
        (col("m.product_name") == col("p.product_name")) &
        (col("m.product_category") == col("p.product_category")) &
        (col("m.product_price") == col("p.product_price")) &
        (col("m.product_quantity") == col("p.product_quantity")) &
        (col("m.pet_category") == col("p.pet_category")) &
        (col("m.product_weight") == col("p.product_weight")) &
        (col("m.product_color") == col("p.product_color")) &
        (col("m.product_size") == col("p.product_size")) &
        (col("m.product_brand") == col("p.product_brand")) &
        (col("m.product_material") == col("p.product_material")) &
        (col("m.product_description") == col("p.product_description")) &
        (col("m.product_rating") == col("p.product_rating")) &
        (col("m.product_reviews") == col("p.product_reviews")) &
        (col("m.product_release_date") == col("p.product_release_date")) &
        (col("m.product_expiry_date") == col("p.product_expiry_date")),
        "left"
    )
    .select(
        col("m.sale_date").alias("sale_date"),
        col("c.id").alias("customer_id"),
        col("s.id").alias("seller_id"),
        col("p.id").alias("product_id"),
        col("m.sale_quantity").alias("sale_quantity"),
        col("m.sale_total_price").alias("sale_total_price")
    )
)

In [ ]:
fact_sales.write.jdbc(
    url=jdbc_url,
    table="fact_sales",
    mode="append",
    properties=connection_props
)